In [27]:
import pyreadr

import sys

import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

import pandas as pd




In [19]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


In [20]:
data = pyreadr.read_r('/Users/jseid1/BSTA6700FinalProject/ScratchWork/outcomes.rds')
data


OrderedDict([(None,
                        0         1         2
              0 -1.733021  5.389870  0.512796
              1 -0.245114  0.759901  3.426867
              2  4.088283 -0.203178 -0.981585
              3  6.776539  1.088276  4.141215
              4  1.659953 -2.167650  5.050269
              5 -2.669239  2.392592 -5.557868)])

- step 1: split into train and test
- step 2: normalize
    - Use the mean and STD computed only from the train data (pre-treatment) - not the test data (post treatment!)
    - apply the standardization to both train and test data
- step 3: figure out how to implement the method on multiple time points.

In [21]:

def learnQ(targets, covariates,embedding_dim,verbose):
    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    l2_norm = sum

    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.01 
        lambda_l2_w = 1
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        
        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final
    

## Toy Example

### Toy data set (no underlying patterns)

Notice that the number of time points is greater than the number of units. What if I change this?

In [22]:

# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0], [4.0, 6.0]], dtype=torch.float64)

d_3 = torch.tensor([4.0, 4.0], dtype=torch.float64)
Y_3 = torch.tensor([[1.0, 3.0], [2.0, 4.0]], dtype=torch.float64)

target_vectors = [d_1, d_2, d_3]
covariate_matrices = [Y_1,Y_2,Y_3]


In [23]:
print(target_vectors[0].shape)   # should be (2,)
print(covariate_matrices[0].shape)  # should be (2, 2)

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 10, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(d_3)
print(Y_3@Q_final@w_final)



torch.Size([2])
torch.Size([2, 2])
Step    0 | Loss: 49.11450626
Step  200 | Loss: 17.00118149
Step  400 | Loss: 15.89905812
Step  600 | Loss: 15.89571876
Step  800 | Loss: 15.64368608
Step 1000 | Loss: 15.60265322
Step 1200 | Loss: 15.60092724
Step 1400 | Loss: 15.59762699
Step 1600 | Loss: 15.59571574
Step 1800 | Loss: 15.59392211
tensor([6., 2.], dtype=torch.float64)
tensor([4.3946, 3.5984], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([2.1973, 2.8021], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([5.1909, 4.3946], dtype=torch.float64)


Making the number of donors greater than the number of time points.
This has the effect of greatly decreasing the loss.
And it is also pretty clear that the loss doesn't depend too strongly on the embedding dimension.

In [24]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)



Step    0 | Loss: 72.60129912
Step  200 | Loss: 1.08127973
Step  400 | Loss: 0.82281019
Step  600 | Loss: 0.67314261
Step  800 | Loss: 0.57412767
Step 1000 | Loss: 0.52426619
Step 1200 | Loss: 0.50280760
Step 1400 | Loss: 0.49256146
Step 1600 | Loss: 0.48647478
Step 1800 | Loss: 0.48221224
tensor([6., 2.], dtype=torch.float64)
tensor([5.9169, 1.9300], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([5.0676, 3.0900], dtype=torch.float64)


# Synthetic Data

In [ ]:
# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_units, n_timepoints, n_outcomes)
Y

     Unnamed: 0  x
0       n_units  5
1  n_timepoints  3
2    n_outcomes  2
5 3 2


array([[[ 1.74001926,  1.92620364],
        [ 1.69194247,  0.31600265],
        [ 2.07379213,  0.48275098]],

       [[ 3.23610359,  2.35926084],
        [-1.13588041,  3.69303016],
        [ 1.63039026,  3.51910486]],

       [[-3.57955365, -0.22815764],
        [ 2.63181248, -3.28550694],
        [ 0.14729029, -0.01496501]],

       [[ 0.33055032,  1.42259774],
        [ 5.23558072,  1.37449261],
        [-4.6987159 ,  1.7540379 ]],

       [[ 2.98518151,  0.19723961],
        [-0.76055506,  2.66526358],
        [ 2.66988027,  3.49555944]]])